# **Joseph Dominic Bichara**
# PCA & Feature Selection
## VascularGuard ML — Dimensionality Reduction & Feature Engineering

---

## 📚 What is this notebook about?

This notebook covers two important techniques that every ML project should include:

**1. PCA (Principal Component Analysis)** — Dimensionality Reduction  
**2. RFE (Recursive Feature Elimination)** — Feature Selection

### Why do we need these?

When you have many features (we have 37 for IV and 39 for Dialysis), some of them:
- Carry **redundant information** (correlated with other features)
- Add **noise** without helping the model
- Make the model **slower to train** and harder to interpret

Both techniques help you understand and reduce the complexity of your data.

---

### Notebook Structure
1. Imports & Setup  
2. PCA — What it is and how it works  
3. PCA — Scree plots (how many components do we need?)  
4. PCA — 2D visualisation (can we see class separation?)  
5. Feature Selection — Correlation filtering  
6. Feature Selection — RFE (Recursive Feature Elimination)  
7. Key Findings


## 1. Imports & Setup

In [2]:
import numpy as np
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

# Load processed data (already scaled — important for PCA)
dl_X_train = np.load('data/processed/dialysis_X_train.npy')
dl_X_test  = np.load('data/processed/dialysis_X_test.npy')
dl_y_test  = np.load('data/processed/dialysis_y_test.npy')
dl_fn      = joblib.load('models/dialysis_models/dialysis_feature_names.pkl')

print(f"Dial X_train: {dl_X_train.shape}")


Dial X_train: (2652, 39)


## 2. What is PCA?

### 🧠 The Concept (Plain English)

Imagine you have 37 features. Instead of thinking of each feature separately, PCA asks:
> *"Can we find a smaller number of directions in the data that capture most of the variation?"*

Each **Principal Component (PC)** is a new axis that:
- Is a **weighted combination** of the original features
- Captures as much variation in the data as possible
- Is **uncorrelated** with all other PCs

**Analogy:** Think of a shadow cast by a 3D object. The shadow is 2D but still captures the object's shape. PCA finds the best "angle" to cast that shadow so you lose as little information as possible.

### ⚠️ Important Rule: Always Scale Before PCA

PCA is sensitive to the scale of features. A feature measured in thousands (like platelet count) would dominate over a 0/1 binary feature if you don't scale first. **We're fine — our data is already StandardScaled.**

### Key Terms:
| Term | Meaning |
|------|---------|
| **Principal Component (PC)** | A new axis (linear combination of original features) |
| **Explained Variance Ratio** | How much of the total variation PC#n captures |
| **Cumulative Explained Variance** | Total variation captured by the first n components |
| **Scree Plot** | Graph showing how much variance each PC explains |


## 3. PCA — Scree Plots

### How many components do we actually need?

The key question in PCA is: *"How many principal components should I keep?"*

**Standard answer:** Keep enough to explain **95% of the variance** in your data.

The scree plot below shows cumulative explained variance. Where the red line (95%) 
intersects the curve is how many components you need.


In [4]:
# Fit full PCA on training data (no target leakage — train only)
pca_dl_full = PCA().fit(dl_X_train)

cum_var_dl = np.cumsum(pca_dl_full.explained_variance_ratio_) * 100

n95_dl = int(np.argmax(cum_var_dl >= 95)) + 1
n99_dl = int(np.argmax(cum_var_dl >= 99)) + 1

print(f"Dialysis Access:")
print(f"  Original features : {len(dl_fn)}")
print(f"  95% variance kept : {n95_dl} components  (saves {len(dl_fn) - n95_dl} features)")
print(f"  99% variance kept : {n99_dl} components")


Dialysis Access:
  Original features : 39
  95% variance kept : 33 components  (saves 6 features)
  99% variance kept : 37 components


## 4. PCA — 2D Visualisation

### What does the data look like in 2D?

By reducing to just 2 principal components, we can **plot the entire dataset** on a 2D graph
and colour the points by class (failure vs. no failure).

**What to look for:**
- If the two colours **cluster separately** → the model should find it easy to classify
- If the colours are **completely mixed** → the signal is subtle and the model has to work harder

**Our finding:** Dialysis captures **10.8%** — meaning the data is genuinely high-dimensional and 
the failure signal is spread across many features, not concentrated in 1-2.


## 5. Feature Selection — Method 1: Correlation Filtering

### 🧠 The Concept

Before throwing all 37 features into a model, we should check: are any features 
**so correlated with each other** that they're basically saying the same thing?

If `feature_A` and `feature_B` have correlation = 0.95, keeping both adds noise without
adding information. We can safely drop one.

**Threshold:** Typically remove one feature from any pair with |correlation| > 0.85.

This is a **filter method** — it looks at the data *before* any model is trained.


In [5]:
import pandas as pd

# Correlation matrix on training data
dl_corr  = pd.DataFrame(dl_X_train, columns=dl_fn).corr().abs()

def find_high_corr_pairs(corr_matrix, threshold=0.85):
    pairs = []
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    for col in upper.columns:
        for row in upper.index:
            val = upper.loc[row, col]
            if val > threshold:
                pairs.append((row, col, round(val, 3)))
    return pairs

dl_pairs  = find_high_corr_pairs(dl_corr)

print(f"Dial — feature pairs with |correlation| > 0.85: {len(dl_pairs)}")
for a, b, v in dl_pairs:
    print(f"  {a}  ↔  {b}  (r={v})")


Dial — feature pairs with |correlation| > 0.85: 0


## 6. Feature Selection — Method 2: RFE (Recursive Feature Elimination)

### 🧠 The Concept

RFE is a **wrapper method** — it actually trains a model repeatedly to figure out which features matter.

**How it works (step by step):**
1. Train a model on all features
2. Ask the model: *"which feature is least important?"*
3. Remove that feature
4. Repeat until you have the desired number of features

It's like elimination rounds on a competition show — the weakest feature gets voted off each round.

**Why Logistic Regression for RFE?** It's fast, and its coefficients directly tell us feature importance.
The L1 penalty (Lasso) naturally drives unimportant coefficients toward zero.

### Results:

**Dialysis RFE selected 15 features:** `peripheral_vascular_disease`, `albumin_level`, `dial_maturation_status`, `dial_blood_flow_rate_ml_min`, `dial_venous_pressure_mmhg`...

**Observation:** RFE's top features for both datasets align well with SHAP rankings from 
notebooks 05/06 — a good sign that the signal is consistent across methods.


In [6]:
# RFE on Dialysis data
lr_dl = LogisticRegression(C=0.1, penalty='l1', solver='liblinear', random_state=42, max_iter=1000)
rfe_dl = RFE(lr_dl, n_features_to_select=15, step=2)
rfe_dl.fit(dl_X_train, np.load('data/processed/dialysis_y_train.npy'))

selected_dl = [dl_fn[i] for i in range(len(dl_fn)) if rfe_dl.support_[i]]
eliminated_dl = [dl_fn[i] for i in range(len(dl_fn)) if not rfe_dl.support_[i]]

print(f"Dialysis RFE — Selected {len(selected_dl)} of {len(dl_fn)} features:")
for f in selected_dl:
    print(f"  ✅ {f}")
print(f"\nEliminated {len(eliminated_dl)} features:")
for f in eliminated_dl:
    print(f"  ❌ {f}")


Dialysis RFE — Selected 15 of 39 features:
  ✅ peripheral_vascular_disease
  ✅ albumin_level
  ✅ dial_maturation_status
  ✅ dial_blood_flow_rate_ml_min
  ✅ dial_venous_pressure_mmhg
  ✅ dial_recirculation_rate_pct
  ✅ dial_kt_v
  ✅ dial_prior_interventions
  ✅ dial_steal_syndrome
  ✅ dial_surgeon_volume
  ✅ dial_central_venous_catheter_history
  ✅ low_flow_flag
  ✅ smoking_status_former
  ✅ dial_access_location_brachiocephalic
  ✅ dial_access_location_upper_arm_graft

Eliminated 24 features:
  ❌ age
  ❌ bmi
  ❌ diabetes
  ❌ hypertension
  ❌ heart_failure
  ❌ anticoagulant_use
  ❌ hemoglobin
  ❌ platelet_count
  ❌ history_of_difficult_access
  ❌ number_previous_access_attempts
  ❌ edema_present
  ❌ dial_access_age_months
  ❌ dial_prior_access_failures
  ❌ dial_time_since_last_intervention_days
  ❌ dial_infection_history
  ❌ dial_months_on_dialysis
  ❌ dial_dominant_arm
  ❌ high_vp_flag
  ❌ sex_M
  ❌ smoking_status_never
  ❌ dial_access_type_av_graft
  ❌ dial_access_location_forearm_graf

## 7. Key Findings

### PCA Summary

| Dataset | Original Features | 95% Variance PCs | Reduction |
|---------|-------------------|-----------------|-----------|
| Dialysis Access | 39 | 33 | 6 features removed |

**Interpretation:** The relatively small reduction (e.g., 37 → 32 for IV) tells us the data 
is genuinely high-dimensional — no single feature dominates, and the variance is spread across many 
dimensions. The 2D scatter confirms this: failure and non-failure patients are **not cleanly separable 
in 2D**, which is why tree-based ensemble models (Random Forest) outperform simpler linear models here.

### Feature Selection Summary

**Dialysis — RFE top features confirm EDA + SHAP findings:**
- `dial_blood_flow_rate_ml_min`, `dial_kt_v`, `dial_maturation_status`, `dial_steal_syndrome` all selected ✅
- `low_flow_flag` (engineered feature) selected — confirms the clinical threshold engineering was worthwhile

### Why We Kept All Features in the Final Models

We chose NOT to drop features based on RFE for the production models. Reasons:
1. 37/39 features is not prohibitively large for Random Forest
2. Dropping features reduces SHAP interpretability coverage
3. RF is robust to uninformative features — it naturally assigns them low importance

RFE serves here as **analytical evidence** for the feature importance narrative, not as a preprocessing step.
